## Loading raw dataset and preparing it to make star schema to store in dataware house

In [0]:
# Importing libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import col, sum as sparkSum

In [0]:
#Initializing spark session
spark = SparkSession.builder.appName("Preprocessor").getOrCreate()

In [0]:
# File location and type
file_location = "/FileStore/tables/fact_sales_data_v2.csv"
file_type = "csv"

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", True) \
  .option("header", True) \
  .load(file_location)

display(df)

ProductCategory,ProductName,Brand,StoreRegion,StoreName,StoreType,SalesRep,Department,EmployeeRole,UnitsSold,UnitPrice,Discount,SaleDate
Furniture,T-shirt,BrandB,East,StoreX,Franchise,Martha Long,Electronics,Cashier,12.0,-1.0,5.0,2022-12-14
Clothing,Tablet,BrandC,East,StoreZ,Franchise,Martha Long,Home,Sales Associate,null,272.49,null,2023-02-24
Clothing,Tablet,BrandA,South,StoreX,Retail,Emily Vazquez,Apparel,Cashier,null,484.75,15.0,2025-03-24
Electronics,Smartphone,BrandB,West,StoreY,Outlet,Charles Fields,Apparel,Cashier,null,205.74,10.0,2023-09-30
Furniture,T-shirt,BrandC,East,StoreZ,Outlet,Wendy Castillo,Home,Manager,46.0,20.25,5.0,2022-10-14
Furniture,T-shirt,BrandC,South,StoreY,Retail,Wendy Castillo,Home,Manager,null,361.06,10.0,2024-02-23
Clothing,T-shirt,BrandC,South,StoreY,Outlet,John Harris,Home,Cashier,37.0,492.65,5.0,2024-05-06
Electronics,Smartphone,BrandC,South,StoreX,Outlet,Charles Fields,Home,Sales Associate,37.0,293.87,15.0,2023-04-04
Clothing,Jeans,BrandA,South,StoreY,Retail,Wendy Castillo,Electronics,Manager,23.0,189.47,15.0,2022-12-26
Furniture,T-shirt,BrandB,East,StoreZ,Franchise,Charles Fields,Apparel,Manager,25.0,359.08,10.0,2022-10-28


<b>Replace "/Users/phuyelgautam3@gmail.com/get_null_columns" with the correct path to your notebook


In [0]:
%run /Users/phuyelgautam3@gmail.com/get_null_columns



<b>Detecting null columns

In [0]:
get_null_columns(df)

Out[88]: ['ProductName', 'UnitsSold', 'Discount']

In [0]:
df.printSchema()

root
 |-- ProductCategory: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- StoreRegion: string (nullable = true)
 |-- StoreName: string (nullable = true)
 |-- StoreType: string (nullable = true)
 |-- SalesRep: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- EmployeeRole: string (nullable = true)
 |-- UnitsSold: double (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- SaleDate: date (nullable = true)



<b>Strategy for null values
<li>ProductName: If null, Unknown
<li>UnitSold: If null, -1
<li>Discount: If null, -1

In [0]:
df_fixed = df.fillna({
    "ProductName": "Unknown",
    "UnitsSold": -1,
    "Discount": -1
    })

In [0]:
#checking null column again if exists
get_null_columns(df_fixed)

Out[91]: []

In [0]:
df_fixed.write \
    .format("delta")\
        .mode("overwrite") \
            .save("/FileStore/tables/Fact_Sales_Preprocessed")